# 11장. 파인튜닝: 요청을 따르게 만들기

이야기를 이어 쓰기만 하던 모델을, **요청을 받으면 그에 맞는 이야기를 들려주는 모델**로 바꿔요. **T4 GPU가 필요하고, 학습에 10분 정도 걸립니다**(본 학습 5분 + 과적합 실험 5분). (런타임 유형: T4 GPU)

**이 장이 끝나면**: 드라이브에 `sft.pt`가 저장됩니다. GPU가 없으면 12장은 저자 체크포인트로 진행할 수 있어요.


### 준비


In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm

import os
import torch
from modu_llm import Vocab, GPTConfig, END, generate, encode_texts
from modu_llm import load_checkpoint, get_file, mount_drive, train
from modu_llm import format_pair, load_sft_texts, get_device, plot_history

device = get_device(need_gpu=True)
vocab = Vocab.build()
folder = mount_drive()

SFT_MINUTES = 5
SFT_LR = 1e-4            # 사전학습보다 작게: 배운 것을 크게 흐트러뜨리지 않으려고
sft_path = f"{folder}/sft.pt"

pre_model, _, pre_step, _ = load_checkpoint(get_file("pretrain.pt"), device)
print(f"사전학습 모델: {pre_step}단계")


## 11.1 사전학습 모델은 왜 요청을 못 따를까

사전학습 모델은 이야기만 읽었어요. "요청: ..."이라는 글은 본 적이 없으니, 요청에 답하는 대신 그 글을 **그럴듯하게 이어 쓰기만** 합니다.

**핵심 코드**: `format_pair(request)` → `"요청: ...\n이야기: "`


In [ ]:
request = "숲속에서 토끼 보리가 나오는 이야기를 들려줘"
question = format_pair(request)
torch.manual_seed(0)
answer = generate(pre_model, vocab, question, max_new_tokens=200)
print(question + answer)


## 11.2 요청-이야기 데이터 살펴보기

요청 문장과 그에 맞는 이야기를 짝지은 데이터예요. 모델은 이 형식을 보고 "요청 다음에는 그에 맞는 이야기가 오고, 이야기가 끝나면 §가 온다"를 배웁니다.

**핵심 코드**: `load_sft_texts("train")`


In [ ]:
texts = load_sft_texts("train")
val_texts = load_sft_texts("validation")
print(f"학습용 {len(texts):,}쌍 | 검증용 {len(val_texts):,}쌍")
for t in texts[:2]:
    print("-" * 40)
    print(t + END)

block_size = GPTConfig().block_size
too_long = 0
for t in texts:
    if len(t) + 1 > block_size:
        too_long = too_long + 1
print()
print(f"문맥 길이({block_size}글자)보다 긴 쌍: {too_long}개")


## 11.3 이 데이터는 어떻게 만들었나

이 데이터는 사람이 하나하나 쓴 것이 아니에요. 저자가 Tiny-Ko-Stories의 이야기 첫머리에서 **주인공 이름, 동물, 장소, 때**를 규칙으로 찾아 요청 문장을 자동으로 만들었습니다.

```
이야기: 햇볕이 쨍쨍한 여름날, 꼬마 토끼 보리가 숲속 모험을 떠나요. ...
찾은 것: 이름=보리, 동물=토끼, 장소=숲속, 때=여름날
요청:   숲속에서 토끼 보리가 나오는 이야기를 들려줘
```

이야기에 실제로 있는 내용만 요청에 넣었기 때문에, 요청과 이야기가 어긋나지 않아요. 같은 방법을 여러분의 데이터에도 쓸 수 있습니다.


## 11.4 저장된 모델에서 이어 학습하기

사전학습 모델을 불러와 요청-이야기 데이터로 **이어서** 학습해요. 학습은 9장과 똑같이 패키지의 `train` 함수가 맡고, 달라진 것은 데이터와 출발점, 학습률뿐입니다.
드라이브에 `sft.pt`가 있으면 그 지점부터 이어 가요. `keep_best=True`는 11.5에서 설명합니다.

**핵심 코드**: `train(..., keep_best=True)`


In [ ]:
train_data = encode_texts(texts, vocab)
val_data = encode_texts(val_texts, vocab)

if os.path.exists(sft_path):
    model, optimizer, step, history = load_checkpoint(sft_path, device, optimizer_lr=SFT_LR)
    print(f"파인튜닝 {step}단계에서 이어 갑니다.")
else:
    model, optimizer, _, _ = load_checkpoint(get_file("pretrain.pt"), device, optimizer_lr=SFT_LR)
    step = 0
    history = []
    print("사전학습 모델에서 파인튜닝을 시작합니다.")

step, history = train(
    model, optimizer, train_data, val_data,
    minutes=SFT_MINUTES,
    ckpt_path=sft_path,
    eval_every=200,
    start_step=step,
    history=history,
    keep_best=True,      # 검증 손실이 가장 낮을 때만 저장
)
print("학습 끝:", step, "단계")


## 11.5 과적합 일으켜 보기

11.4의 손실은 처음 1분 안에 크게 내려간 뒤 거의 평평해요. 4만 쌍이 넘는 데이터를 몇 바퀴 읽는 정도로는 모델이 외우지 않기 때문입니다.
데이터가 적으면 어떻게 될까요? 요청-이야기 쌍을 **3,000개만** 떼어, 사전학습 모델에서 11.4와 똑같이 `SFT_MINUTES`분 동안 학습해 봅니다. 이 실험의 결과는 Colab의 임시 폴더에 저장하므로 드라이브의 `sft.pt`는 그대로예요.

**핵심 코드**: `load_sft_texts("train", limit=3000)`


In [ ]:
small_texts = load_sft_texts("train", limit=3000)
small_data = encode_texts(small_texts, vocab)
small_model, small_opt, _, _ = load_checkpoint(get_file("pretrain.pt"), device, optimizer_lr=SFT_LR)

_, small_history = train(
    small_model, small_opt, small_data, val_data,
    minutes=SFT_MINUTES,
    ckpt_path="sft_small.pt",
    eval_every=200,
    keep_best=True,
)


학습 손실은 계속 내려가는데, 검증 손실은 **바닥을 찍고 다시 올라가요**. 3,000쌍을 여러 바퀴 읽는 사이 모델이 이야기를 외우기 시작한 거예요. 9장에서 말로만 본 **과적합**입니다.
그래서 `keep_best=True`로 학습했어요. 검증 손실이 지금까지 중 가장 낮을 때만 저장하므로, 과적합이 일어나도 **가장 좋았던 순간의 모델**이 파일에 남습니다.


In [ ]:
best = small_history[0]
for h in small_history:
    if h["val"] < best["val"]:
        best = h

plot_history(small_history, mark_step=best["step"])      # 점선: 가장 좋았던 곳

last = small_history[-1]
print(f"검증 손실이 가장 낮았던 곳: {best['step']}단계 (검증 {best['val']:.3f}) → 이때 저장")
print(f"학습이 끝난 곳: {last['step']}단계 (검증 {last['val']:.3f})")


## 11.6 같은 요청, 달라진 대답

같은 요청을 사전학습 모델과 파인튜닝 모델(11.4)에 넣어 비교해요. 이 장의 핵심 장면입니다.


In [ ]:
requests = ["숲속에서 토끼 보리가 나오는 이야기를 들려줘",
            "겨울날 놀이터에서 일어난 이야기를 써 줘",
            "강아지 뭉치가 나오는 이야기를 만들어 줘"]
for r in requests:
    print("요청:", r)
    question = format_pair(r)
    for name, m in [("사전학습", pre_model), ("파인튜닝", model)]:
        torch.manual_seed(0)
        answer = generate(m, vocab, question, max_new_tokens=250)
        print(f"  [{name}]", answer.replace("\n", " "))
    print("=" * 40)


## 놀이터

본 모델을 건드리지 않는 활동입니다. 건너뛰어도 다음 장에 영향이 없습니다.

데이터에 없었을 법한 조합을 요청해 보세요. 예: "바닷가에서 다람쥐가 나오는 이야기를 들려줘", "밤 도서관에서 일어난 이야기를 써 줘"


In [ ]:
r = "바닷가에서 다람쥐가 나오는 이야기를 들려줘"
answer = generate(model, vocab, format_pair(r), max_new_tokens=300)
print(answer)


## 실제 LLM에서는

- 실제 파인튜닝은 손실을 **대답 부분에서만** 계산하고, 대화 한 쌍씩 잘라 학습해요. 이 책은 9장 코드를 그대로 쓰려고 모든 글자에서 손실을 계산했습니다.
- 대화 형식에는 `요청:` 같은 글자 대신 전용 특수 토큰을 쓰고, 그 뒤 사람의 선호를 반영하는 학습(RLHF, DPO 등)을 더 거쳐요.
